In [1]:
import torch
from model.model import EncoderDecoderDAG
from utils.data import TranslateDataset, collate_fn, process_data
from torch.utils.data import DataLoader
from utils.load_tokenizer import load_tokenizer
from torch.optim import Adam
from typing import Tuple
from utils.fix_probs import fix_probs, masking
from tqdm.notebook import tqdm
from matplotlib import pyplot as plt
from utils.checkpoint import try_loading, epoch_resume, save_checkpoint
from utils.decoding import greedy_decoding, lookahead

g:\Projects\Visual Studio Code\LMTests\lmtest\lib\site-packages\transformers\utils\hub.py:123: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
tokenizer, vocab_size = load_tokenizer()

In [3]:
pad_idx = tokenizer.pad_token_id
eos_idx = tokenizer.eos_token_id

In [4]:
factor = 4
emb_size = 256
num_heads = 8
max_seq_len = 100
max_vertices = max_seq_len * factor

In [5]:
layers = [(1,1), 1, (1,1)]
out_layers = 1

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [7]:
device

device(type='cuda')

In [8]:
def create_model_fallback_fn() -> Tuple[EncoderDecoderDAG, Adam]:
    model = EncoderDecoderDAG(vocab_size, emb_size, num_heads, max_seq_len, max_vertices, layers, out_layers)
    model.to(device)
    optm = Adam(model.parameters(), lr=1e-3)
    return model, optm
    

In [9]:
checkpoint_dir = "./checkpoints"
checkpoint_name = "naivedag.pt"

In [10]:
model_class = EncoderDecoderDAG
optm_class = Adam

In [11]:
model, optm, losses, log_dir, tokens_seen = try_loading(checkpoint_dir, checkpoint_name, model_class, optm_class, device, create_model_fallback_fn)

Resuming, have seen 2,722,595 epochs and 961,159,205 tokens
Have 102093818 trainable parameters
Logging to runs/run_at_2023-12-29_15-32-10_new_loss


In [48]:
en_test = "<s>Hello, how are you?"

In [49]:
encoded = tokenizer(en_test, return_tensors="pt").input_ids.to(device)

In [50]:
encoded

tensor([[65001,  1496, 26607,     2,     7, 10828,    44,    37,    23,     0]],
       device='cuda:0')

In [51]:
encoded = tokenizer(en_test, return_tensors="pt").input_ids.to(device)
batch_size, l = encoded.shape
decoder_tokens = torch.arange(0, l * factor).unsqueeze(0).expand(batch_size, -1).to(device)
target_lens, vertex_lens, token_mask, vertex_mask = process_data(encoded, pad_idx, factor)
log_transition_probs, log_emission_probs = model(encoded, decoder_tokens, token_mask, vertex_mask, vertex_lens)
# mask = masking(log_transition_probs, vertex_lens)

In [52]:
#flog_transition_probs = log_transition_probs.masked_fill(mask !=0, float('-inf'))
# flog_transition_probs = fix_probs(log_transition_probs, mask)
b1_transitions = log_transition_probs[0]
b1_emissions = log_emission_probs[0]

In [53]:
decoded = greedy_decoding(b1_transitions, b1_emissions, eos_idx)

In [54]:
tokenizer.decode(decoded)

'<s>,,,,,吗?</s>'

In [55]:
decoded2 = lookahead(b1_transitions, b1_emissions, eos_idx)

In [56]:
tokenizer.decode(decoded2)

'<s>,,,,,吗?</s>'